# Diabetes EDA: Step-by-Step Exploratory Data Analysis

This notebook explores a diabetes-related health dataset to understand the patterns behind diabetes risk before building any machine learning model.

In EDA, we try to answer questions like:
- What does the dataset look like?
- Are there missing values?
- How is the target variable distributed?
- Which features show different patterns in people with and without diabetes?
- Which variables look most important for future modeling?

This notebook is designed for learning and interpretation, not just for running code. Each section includes short explanations.


## 1) Import libraries

We start by loading the libraries we need for data manipulation and visualization.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')


## 2) Load the diabetes dataset

The dataset is loaded from a CSV file. If it is not present locally, the notebook downloads it automatically from the reference project.


In [ ]:
data_path = 'data/diabetes_binary_health_indicators_BRFSS2015.csv'

if not os.path.exists(data_path):
    os.makedirs('data', exist_ok=True)
    url = 'https://raw.githubusercontent.com/eviekenna/FinalProjectST558/main/diabetes_binary_health_indicators_BRFSS2015.csv'
    import urllib.request
    urllib.request.urlretrieve(url, data_path)
    print(f'Downloaded dataset to {data_path}')
else:
    print(f'Using local dataset: {data_path}')

df = pd.read_csv(data_path)
df.head()


## 3) Understand the dataset structure

Before analysis, we check the size of the dataset and the column names. This gives us an overview of what information is available and how many rows and features we have.


In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
for i, col in enumerate(df.columns, 1):
    print(f'{i}. {col}')

df.info()


### Interpretation

The dataset has many rows and a moderate number of variables. In this dataset, most features are binary indicators (e.g., `HighBP`, `Smoker`, `PhysActivity`) or categorical ordinal variables such as `Age`, `Education`, and `Income`. This means the dataset is very suitable for EDA because many patterns are easy to summarize and plot.


## 4) Rename and label variables for readability

Some variables are coded as numeric values such as 0 and 1. To make the analysis easier to interpret, we convert them into meaningful labels like 'No Diabetes' and 'Prediabetes/Diabetes'.


In [ ]:
df = df.copy()

df['Diabetes_binary'] = df['Diabetes_binary'].map({
    0: 'No Diabetes',
    1: 'Prediabetes/Diabetes'
})

binary_mapping = {
    'HighBP': {0: 'No High BP', 1: 'High BP'},
    'HighChol': {0: 'No High Chol', 1: 'High Chol'},
    'CholCheck': {0: 'No Check', 1: 'Check'},
    'Smoker': {0: 'Non-Smoker', 1: 'Smoker'},
    'Stroke': {0: 'No Stroke', 1: 'Stroke'},
    'HeartDiseaseorAttack': {0: 'No', 1: 'Yes'},
    'DiffWalk': {0: 'No Difficulty', 1: 'Difficulty'},
    'PhysActivity': {0: 'No Activity', 1: 'Activity'},
    'Sex': {0: 'Female', 1: 'Male'},
    'Fruits': {0: 'No Fruit Intake', 1: 'Fruit Intake'},
    'Veggies': {0: 'No Veg Intake', 1: 'Veg Intake'},
    'HvyAlcoholConsump': {0: 'No Heavy Drinking', 1: 'Heavy Drinking'},
    'AnyHealthcare': {0: 'No Healthcare', 1: 'Healthcare Access'},
    'NoDocbcCost': {0: 'No Cost Barrier', 1: 'Cost Barrier'},
    'GenHlth': {1: 'Excellent', 2: 'Very Good', 3: 'Good', 4: 'Fair', 5: 'Poor'},
    'MentHlth': None,
    'PhysHlth': None,
    'Age': {1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39', 5: '40-44', 6: '45-49', 7: '50-54', 8: '55-59', 9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79', 13: '80+'},
    'Education': {1: 'Never attended', 2: 'Elementary', 3: 'Some HS', 4: 'HS grad', 5: 'Some college', 6: 'College grad'},
    'Income': {1: '<10k', 2: '10-15k', 3: '15-20k', 4: '20-25k', 5: '25-35k', 6: '35-50k', 7: '50-75k', 8: '>75k'}
}

for col, mapping in binary_mapping.items():
    if col in df.columns and mapping is not None:
        df[col] = df[col].map(mapping)

df.head()


## 5) Check for missing data

Real-world datasets often contain missing values. We check for them before doing deeper analysis.


In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]


### Interpretation

If the result is empty, there are no missing values. That is good because it reduces the need for preprocessing. In many public health datasets, this can happen because the dataset is already cleaned.


## 6) Explore the target variable

The target is `Diabetes_binary`, which tells us whether a person has diabetes or not. We check the class balance because this affects model evaluation later.


In [ ]:
target_summary = df['Diabetes_binary'].value_counts().rename_axis('Diabetes Status').reset_index(name='Count')
target_summary['Proportion'] = target_summary['Count'] / target_summary['Count'].sum()
target_summary

sns.countplot(data=df, x='Diabetes_binary', palette='Set2')
plt.title('Target Variable Distribution')
plt.xlabel('Diabetes Status')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.show()


### Interpretation

This is an important concept in classification problems: the target is often imbalanced. If most observations are 'No Diabetes', then a naive model that always predicts 'No Diabetes' can look surprisingly accurate. This is why accuracy alone can be misleading. We must later consider metrics such as precision, recall, F1-score, or class weights.


## 7) Numerical feature summary

For numeric variables like BMI, MentHlth, and PhysHlth, we look at the distribution using descriptive statistics.


In [ ]:
numeric_cols = ['BMI', 'MentHlth', 'PhysHlth', 'Age']
df[numeric_cols].describe().T


### Interpretation

The summary gives us values such as mean, median, minimum, and maximum. For example, BMI usually has a wide range because some individuals are much heavier than others. Variables like `MentHlth` and `PhysHlth` can be highly skewed, which is an important sign that the data distribution is not perfectly normal. Skewed features often need special treatment during modeling.


## 8) Explore age and BMI distributions

A good EDA workflow starts with distribution plots. These help us understand whether the data is balanced or skewed, and where most observations lie.


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['BMI'], bins=40, kde=True, color='steelblue')
plt.title('Distribution of BMI')
plt.xlabel('BMI')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
age_order = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']
sns.countplot(data=df, x='Age', order=age_order, palette='magma')
plt.title('Age Group Distribution')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Interpretation

BMI is often centered around a moderate range, but the distribution may stretch to high values. This can indicate substantial differences between individuals. The age distribution shows how many people belong to each age band, which is useful for understanding the population being studied.


## 9) Compare distributions by diabetes status

Now we ask: do people with diabetes have different BMI or health patterns compared to people without diabetes?


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x='BMI', hue='Diabetes_binary', fill=True, alpha=0.4, common_norm=False, palette='Set2')
plt.title('BMI Distribution by Diabetes Status')
plt.xlabel('BMI')
plt.ylabel('Density')
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='Diabetes_binary', y='BMI', palette='Set2')
plt.title('BMI by Diabetes Status')
plt.xlabel('Diabetes Status')
plt.ylabel('BMI')
plt.show()


### Interpretation

If people with diabetes tend to have higher BMI values, the density and boxplot will shift to the right. This does not prove causation, but it shows that BMI is associated with diabetes status in the dataset.


## 10) Examine risk factors with cross-tabulations

Binary health features such as high blood pressure and high cholesterol are often strong diabetes indicators. We can compare them across groups using counts and proportions.


In [ ]:
risk_features = ['HighBP', 'HighChol', 'Smoker', 'PhysActivity', 'DiffWalk', 'Stroke', 'HeartDiseaseorAttack']

for feature in risk_features:
    print(f'\n=== {feature} ===')
    table = pd.crosstab(df[feature], df['Diabetes_binary'])
    print(table)
    print('\nRow percentages:')
    print(table.div(table.sum(axis=1), axis=0).round(3) * 100)


### Interpretation

Cross-tabulations help us see whether a feature appears more often in people with diabetes. For example, high blood pressure and high cholesterol often show stronger prevalence among the diabetes group. This suggests these features may be valuable for modeling.


## 11) Check high blood pressure and cholesterol patterns

These are classic risk factors and will often stand out in a diabetes EDA.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='HighBP', hue='Diabetes_binary', palette='Set2', ax=axes[0])
axes[0].set_title('High Blood Pressure vs Diabetes Status')
axes[0].set_xlabel('High Blood Pressure')
axes[0].set_ylabel('Count')

sns.countplot(data=df, x='HighChol', hue='Diabetes_binary', palette='Set2', ax=axes[1])
axes[1].set_title('High Cholesterol vs Diabetes Status')
axes[1].set_xlabel('High Cholesterol')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


### Interpretation

If the diabetes group has noticeably higher proportions of people with high blood pressure and high cholesterol, then these variables are likely useful predictors. This is one of the clearest patterns seen in health-related EDA.


## 12) Explore correlation among numeric features

Even though many variables are categorical, numeric health indicators can still be examined with a correlation matrix.


In [ ]:
numeric_df = df.select_dtypes(include=['number'])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', annot=False, fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()


### Interpretation

The heatmap helps identify whether some variables move together. Strong relationships between predictors may signal multicollinearity, which can affect certain models. EDA does not have to solve every modeling problem, but it helps us spot patterns early.


## 13) Summary of findings

At this point, we can start forming hypotheses before building any models.

### Likely insights from the EDA
- The target class is imbalanced, so accuracy alone is not a sufficient performance metric.
- BMI appears to differ by diabetes status, with higher values often associated with the diabetes group.
- High blood pressure and high cholesterol seem strongly related to diabetes risk.
- Age, physical activity, and health behavior variables may also influence the target.
- The dataset is mostly clean and does not appear to have serious missing-value issues.

These insights are useful for choosing the next step: model training and evaluation.


## 14) Next step: prepare for machine learning

Before building models, the usual preparation is to:
- separate features and target
- encode categorical predictors if needed
- split into train and test sets
- select metrics like precision, recall, F1-score, and ROC-AUC
- consider class imbalance handling

This notebook stops at EDA because the purpose here is understanding the dataset and discovering relationships before modeling.
